# Complete ten-case C/D two-stage validation
Let Drive sync, select T4 or L4 GPU, then Run all four code cells. Send the final Latest report.
The current pipeline is frozen: same Qwen checkpoint, prompts, retrieval and generation budgets. Reuse six classification and six explanation attempts, including failures. Run the remaining fourteen classifications and up to fourteen explanations (at most 28 new calls).
This is development validation, not a final held-out test. No LLM judge runs. Semantic judge scores remain exploratory and are not treated as validated grounding measures.
The notebook checks that the twelve existing attempts are synced before allowing inference. Reruns reuse saved attempts without automatic retries.


In [1]:
from pathlib import Path
import os, sys, json, hashlib, subprocess
from google.colab import drive
ROOT = Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
if not Path('/content/drive/MyDrive').is_dir():
    try:
        drive.mount('/content/drive')
    except ValueError as exc:
        raise RuntimeError('Drive connection failed. Select Runtime > Disconnect and delete runtime, reconnect and rerun this cell. Complete Google authorisation. No diagnosis has started.') from exc
if not (ROOT / 'Files/phm_two_stage.py').is_file():
    raise FileNotFoundError(f'Drive is connected but the updated project is missing at {ROOT}. Finish sync or locate CP5403 in the Files panel. Do not force-remount.')
os.chdir(ROOT)
EXPECTED = {'phm_two_stage.py': 'b2fe20ccb2a64dea7ab517320aec919826bff2125aeffc3ab1a28ef0393c8121', 'test_phm_two_stage.py': '9b92bde11d965817c16c9459e2b9cf89b6a9ad93467bd2216a4eade9e987d829', 'phm_filtered_selection_pilot.py': 'f154c9c522426007b934f7ea21f43b51f662cb83942163f7770a2426016baf0a', 'phm_selection_pilot.py': '3eaef1b55c56a3b6a2977d564b54cc0ebdd0deab1548680072c2f569c29eeafb', 'phm_condition_d.py': 'e22457642e68013b1cfd4201bb6951630711d7d4890fdb28659d70dafa7a80c6', 'phm_evidence_selection.py': 'bb8cd412496f3d055d11b6d0f4ac47e970f24d43b4c19493d9ff57fca8b4fac9', 'phm_literature_index.py': '725709141868ae494f1c515f67df5aec4451c0d137d461b3991c7ba436fb7f37', 'phm_diagnosis.py': 'a0b2af61d7d1a1c16512359f1e8029aae48380fc4b698c188f1f8caacf14b2fd', 'phm_diagnosis_qwen.py': 'f3c9938f7fd2638de842139db5f5d2716c57f4839860291279e45c4d47e049bb', 'phm_output_schema.py': '01913f0c804300723c841d7c689be5105749c1e3d9c0bc110d7a9b204ea048e9', 'phm_run_reporting.py': 'fa54efc1c88773ca86e0e1a2343ff83dc34fff5a3d2545dd0a8c9e51415da2a2', 'literature_retrieval_policy_v1.json': '728102ca47c5074f43f7c2ea37a19bbcf0d9a91d896dde530ccf77b8d522ab33', 'requirements-literature-colab.txt': '9dbbad4b5ea0136c7c6b0b4a0b0f309a62ced876b9ed9a56f662c5d593d7755c', 'experiment_config.json': '563acb189f79b8f6335900a164502d01895409bee9747ee6a0f70073c63e9b3b'}
for name, expected in EXPECTED.items():
    path = ROOT / 'Files' / name
    assert path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == expected, f'File not fully synced: {name}'
INDEX = ROOT / 'Datasets/PHM15-literature-D-approved-v1'
manifest = json.loads((INDEX / 'index_manifest.json').read_text(encoding='utf-8'))
for name, key in [('approved_chunks.jsonl','chunks_sha256'), ('embeddings.npy','embeddings_sha256'), ('model_assets/minilm.onnx','onnx_sha256')]:
    assert hashlib.sha256((INDEX / name).read_bytes()).hexdigest() == manifest[key], f'Index not fully synced: {name}'
print('Project and approved index verified:', manifest['chunk_count'], 'chunks')


Mounted at /content/drive
Project and approved index verified: 24 chunks


In [2]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Files/requirements-literature-colab.txt', 'onnxruntime==1.31.1', 'lm-format-enforcer==0.11.3'], check=True)
print('Dependencies installed.')


GPU: NVIDIA L4
Dependencies installed.


In [3]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'Files', '-p', 'test_phm_two_stage.py', '-v'], check=True)
sys.path.insert(0, str(ROOT/'Files'))
import phm_two_stage as pipeline
cfg = json.loads((ROOT/'Files/experiment_config.json').read_text(encoding='utf-8'))
cfg['generator'].update(max_input_tokens=6144, max_new_tokens=384)
_, records, requests, _ = pipeline.prepare(ROOT, cfg, 10)
cached_decisions = cached_explanations = 0
for request, record in zip(requests, records):
    check, response = pipeline.inspect(ROOT/'Outputs/diagnosis/two_stage_v1/decision'/(request['request_id']+'.json'), request, record)
    cached_decisions += response is not None and check['status'] != 'invalid_cache'
    if check['status'] == 'valid':
        explanation = pipeline.explanation_request(request, check['parsed'])
        echeck, eresponse = pipeline.inspect(ROOT/'Outputs/diagnosis/two_stage_v1/explanation'/(explanation['request_id']+'.json'), explanation, {**record, '_request':explanation})
        cached_explanations += eresponse is not None and echeck['status'] != 'invalid_cache'
assert cached_decisions >= 6 and cached_explanations >= 6, 'Existing attempts are missing or invalid. Finish syncing Outputs before continuing; do not regenerate the baseline.'
print('Cached classifications:', cached_decisions, 'Cached explanations:', cached_explanations)
print('At most', 40-cached_decisions-cached_explanations, 'new calls. Failed cached attempts are retained.')


Cached classifications: 6 Cached explanations: 6
At most 28 new calls. Failed cached attempts are retained.


In [4]:
subprocess.run([sys.executable, 'Files/phm_two_stage.py', '--root', str(ROOT), '--max-cases', '10'], check=True)
latest = json.loads((ROOT/'Outputs/LATEST_RUN.json').read_text(encoding='utf-8'))
print('Latest report:', latest)
print('STOP: let Drive sync and send the complete ten-case C/D report. No judge calibration or diagnosis retries are needed.')


Latest report: {'run_id': '20261122T091139_paired_CD_two_stage_validation_114167e3', 'run_directory': 'Outputs/runs/20261122T091139_paired_CD_two_stage_validation_114167e3', 'status': 'completed_with_issues', 'updated_utc': '2026-11-22T09:29:38.920730+00:00'}
STOP: let Drive sync and send the complete ten-case C/D report. No judge calibration or diagnosis retries are needed.
